# FINS 实验流水线

一条 notebook 串起 4 个模块（原 notebook 已 py 化）：

| 步骤       | 模块 | 作用                                                                                           |
|------------|---|------------------------------------------------------------------------------------------------|
| 1 生成配置 | `_load.py` | 按拓扑 / 时序 / 负载随机生成 pipeline cfg → `pipeline/*.json`                             |
| 2 采集     | `_test.py` | 【fins 测试专用】每份 cfg 起 `bin/client` + `bin/server` 跑 `dur_s` 秒 → trace 复制到 `result/`              |
| 3 标准化   | `std.py` | 【fins 测试专用】 用 JSON 的执行用时在 `execute→complete` 内插 `working` 行 → `result_std/` |
| 4 分析画图 | `plot.py` | 核心甘特、核利用率、生命周期分布                                                               |

> 三个数据目录都在**仓库根**：`pipeline/`（配置）、`result/`（原始 trace）、`result_std/`（标准化 trace）。
> 下面所有相对路径都按仓库根解析，不依赖 notebook 的 cwd。

In [5]:
import importlib

# 导入你的基础模块与工具库
import _load
import _test
import _lttng2csv
# import _csv2graph

# 导入绘图模块
import _csv2overhead
import _csv2timeline
from _csv2overhead import analyze_cpu_utilization
from _csv2timeline import generate_execution_gantt

# 将所有需要支持热重载的本地模块统一放到元组中刷新
for m in (_load, _test, _csv2timeline, _csv2overhead):
  importlib.reload(m)

## 1. 生成配置（`_load.py`）

5 种拓扑（multihop / fork / join / feedback / mixed，均可混入 acc 的 hist 窗口读）
+ 时序（`ptimed`、`period_divisors`）+ 负载（`u`、`H_ms` 等）随机生成，
文件名 `<kind>_u<u>_m<m>_ms<桶>_s<seed>.json`。

In [ ]:
my_cfg_dir = "tool/pipeline"
CONFIG = {

    "topology": {

        "multihop": {
            "paths": (1, 4),
            "depth": (2, 8),
        },

        "fork": {
            "fan": (2, 8),
            "bdepth": (1, 3),
        },

        "join": {
            "fan": (2, 8),
            "bdepth": (1, 3),
            "tail": (1, 3),
        },

        "feedback": {
            "depth": (3, 8),
            "histN": (3, 8),
        },

        "mixed": {
            "nseg": (3, 8),

            "chain_prob": 0.45,
            "fork_join_prob": 0.30,
            "feedback_prob": 0.25,
        },
    },

    "temporal": {

        # Probability that a non-source node is timed.
        "ptimed": 0.35,

        # T values are H / divisor.
        #
        # Example:
        #
        # H=100
        #
        # divisor 1 -> 100 ms
        # divisor 2 -> 50 ms
        # divisor 4 -> 25 ms
        #
        "period_divisors": [1, 2, 4, 5, 10, 20],
    },

    "workload": {

        "u": [
            0.1,
            0.3,
            0.7,
            0.9,
        ],

        "workers": [
            1,
            2,
            3,
        ],

        # Hyperperiod.
        "H_ms": 100,

        # ----------------------------------------------------
        # Makespan classification.
        #
        # Example H=100, width=5:
        #
        # bin 00: [0,5)
        # bin 01: [5,10)
        # ...
        # bin 18: [90,95)
        # bin 19: [95,100]
        #
        # The last bucket includes H.
        # ----------------------------------------------------

        "makespan_bin_width_ms": 10.0,

        # Anything above this goes into overflow.
        #
        # None:
        #     use H_ms.
        #
        "makespan_max_ms": 80,

        # Number of final samples per bucket.
        "n_per": 5,

        # ----------------------------------------------------
        # Generation policy.
        # ----------------------------------------------------

        # Initial candidate generation.
        "initial_attempts": 500,

        # Extra attempts for incomplete buckets.
        "refill_attempts": 500,

        # Maximum number of refill rounds.
        "max_refill_rounds": 1,

        # Keep at most:
        #
        #     n_per * candidate_factor
        #
        # candidates per bucket.
        #
        "candidate_factor": 1,

        # Random seed.
        "seed_base": 20260910,

    },

    "solver": {

        # Utilization numerical tolerance.
        "u_tolerance": 1e-8,

        # Minimum WCET.
        #
        # Plugin cfg is in microseconds.
        #
        "min_wcet_us": 100,

        # C_i <= T_i * max_c_ratio
        "max_c_ratio": 1.0,

        # Number of attempts used to find a C allocation.
        "c_attempts": 300,
    },
}

_load.generate_all(
    out_dir=my_cfg_dir,
    config=CONFIG
)

## 2. 采集（`_test.py`）

对每份 cfg：起 `client` + 发配置 + 跑 `dur_s` 秒 → 终止 → 把 `tool/temp/tracing.csv`
复制成 `result/<cfg名>.csv`（原件保留，只搬原始 trace，不算指标）。

正式实验要走独占核：`cores="1-6"` 会改用 `sudo tool/client.sh <cores> <workers>`（需要 root）。

In [6]:
# 指定你的目录
my_cfg_dir = "tool/pipeline"
my_result_dir = "tool/result_FINS"

# 一键运行（cpu_offset=1 代表核心从 1 开始排，如果 m=3 就会自动分配核 "1-3"）
_test.run_all(
    target_cfg_dir=my_cfg_dir,
    target_result_dir=my_result_dir,
    run_time=10.0,     # 运行时间
    cpu_offset=1      # 起始核号（避开核心 0 给系统）
)

🔒 该评测脚本需要 root 权限来配置 Cgroup 与绑定核心
🚀 开始智能批量评测（自动从文件名匹配 m）
   📂 输入配置目录: tool/pipeline
   📁 结果输出目录: tool/result_FINS
   📊 发现测试用例: 5 个

进度 [1/5]: feedback_u70_m3_ms05_s20632672.json

>>> [开始测试] feedback_u70_m3_ms05_s20632672 (自动解析: m=3 -> workers=3, cores=1-3)
  [1/5] 创建并启动 LTTng 追踪会话 (fins_eval_feedback_u70_m3_ms05_s20632672_213624)...
  [2/5] 等待 LTTng 预热 1.0s 并推送配置...
  [2/5] 启动 Client 进程 (裸跑模式, workers=3)...
  [3/5] 等待客户端预热 1.0s 并推送配置...
  [Server] 配置灌入成功，持续运行 10.0s...
  [4/5] 停止并销毁 LTTng 追踪会话...
  [5/5] 关闭清理 Client 进程 (pgid=36771)...
  [成功] 轨迹数据已归档至: /home/jenny/Documents/GitHub/fins/tool/result_FINS/feedback_u70_m3_ms05_s20632672_213624/trace
  [验证] ✅ 成功捕捉到 15643 条目标跟踪事件！

进度 [2/5]: fork_u70_m3_ms05_s20984935.json

>>> [开始测试] fork_u70_m3_ms05_s20984935 (自动解析: m=3 -> workers=3, cores=1-3)
  [1/5] 创建并启动 LTTng 追踪会话 (fins_eval_fork_u70_m3_ms05_s20984935_213640)...
  [2/5] 等待 LTTng 预热 1.0s 并推送配置...
  [2/5] 启动 Client 进程 (裸跑模式, workers=3)...
  [3/5] 等待客户端预热 1.0s 并推送配置...
  [Server] 配置灌入成功，持续

# 3. 转义（tran）
1. lttng 2 timeline
2. lttng 2 overhead

In [7]:
_lttng2csv.run_export(
    results_dir="./result_FINS",
    outdir="./result_FINS",
    after_us=2000 * 1000,
    preempt_all=True,
    include_cpus=_lttng2csv.cpus(cpu_start=1, num_workers=3),
)


开始处理 5 个实验 -> ./result_FINS/  输出: timeline, preempt
  [成功] feedback_u70_m3_ms05_s20632672_213624  t0=11017781043900  ->  feedback_u70_m3_ms05_s20632672_213624_timeline.csv (9306 行) | feedback_u70_m3_ms05_s20632672_213624_preempt.csv (5740 行)
  [成功] fork_u70_m3_ms05_s20984935_213640  t0=11034076942886  ->  fork_u70_m3_ms05_s20984935_213640_timeline.csv (7585 行) | fork_u70_m3_ms05_s20984935_213640_preempt.csv (14290 行)
  [成功] join_u70_m3_ms07_s20628568_213657  t0=11050988044384  ->  join_u70_m3_ms07_s20628568_213657_timeline.csv (21401 行) | join_u70_m3_ms07_s20628568_213657_preempt.csv (7026 行)
  [成功] mixed_u70_m3_ms05_s30924190_213714  t0=11067631817784  ->  mixed_u70_m3_ms05_s30924190_213714_timeline.csv (11937 行) | mixed_u70_m3_ms05_s30924190_213714_preempt.csv (16814 行)
  [成功] multihop_u70_m3_ms05_s20633261_213731  t0=11085138142708  ->  multihop_u70_m3_ms05_s20633261_213731_timeline.csv (8757 行) | multihop_u70_m3_ms05_s20633261_213731_preempt.csv (23729 行)
完成。成功 5 / 5


## 4. 分析与画图（`plot.py`）



In [8]:
# CIE_FIFO_IPC_nuc12-noturbo_hythread
# feedback_u70_m3_ms05_s20632672_195219_preempt.csv
# feedback_u70_m3_ms05_s20632672_195219_timeline.csv
fig_gantt_ROS_7B = generate_execution_gantt(
    preempt_csv = "typical/CIE_FIFO_IPC_nuc12_7B/feedback_u70_m3_ms05_s20632672_195219_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_IPC_nuc12_7B/feedback_u70_m3_ms05_s20632672_195219_timeline.csv",
    zoom_window_ms=[0, 1000])

# CIE_FIFO_IPC_nuc12_1kB
# feedback_u70_m3_ms05_s20632672_152109_preempt.csv
# feedback_u70_m3_ms05_s20632672_152109_timeline.csv
fig_gantt_ROS_1kB = generate_execution_gantt(
    preempt_csv = "typical/CIE_FIFO_IPC_nuc12_1kB/feedback_u70_m3_ms05_s20632672_152109_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_IPC_nuc12_1kB/feedback_u70_m3_ms05_s20632672_152109_timeline.csv",
    zoom_window_ms=[0, 1000])

# CIE_FIFO_IPC_nuc12_1MB
# feedback_u70_m3_ms05_s20632672_152938_preempt.csv
# feedback_u70_m3_ms05_s20632672_152938_timeline.csv
# fig_gantt_ROS_1MB = generate_execution_gantt(
#     preempt_csv = "typical/CIE_FIFO_IPC_nuc12_1MB/feedback_u70_m3_ms05_s20632672_152938_preempt.csv",
#     timeline_csv = "typical/CIE_FIFO_IPC_nuc12_1MB/feedback_u70_m3_ms05_s20632672_152938_timeline.csv",
#     zoom_window_ms=[0, 1000])

# FINS_7B
# feedback_u70_m3_ms05_s20632672_210928_preempt.csv
# feedback_u70_m3_ms05_s20632672_210928_timeline.csv
fig_gantt_FINS_7B = generate_execution_gantt(
    preempt_csv = "typical/FINS_7B/feedback_u70_m3_ms05_s20632672_210928_preempt.csv",
    timeline_csv = "typical/FINS_7B/feedback_u70_m3_ms05_s20632672_210928_timeline.csv",
    zoom_window_ms=[0, 1000])

# FINS_1kB
# feedback_u70_m3_ms05_s20632672_211315_preempt.csv
# feedback_u70_m3_ms05_s20632672_211315_timeline.csv
fig_gantt_FINS_1kB = generate_execution_gantt(
    preempt_csv = "typical/FINS_1kB/feedback_u70_m3_ms05_s20632672_211315_preempt.csv",
    timeline_csv = "typical/FINS_1kB/feedback_u70_m3_ms05_s20632672_211315_timeline.csv",
    zoom_window_ms=[0, 1000])

# FINS_1MB
# feedback_u70_m3_ms05_s20632672_213624_preempt.csv
# feedback_u70_m3_ms05_s20632672_213624_timeline.csv
fig_gantt_FINS_1MB = generate_execution_gantt(
    preempt_csv = "typical/FINS_1MB/feedback_u70_m3_ms05_s20632672_213624_preempt.csv",
    timeline_csv = "typical/FINS_1MB/feedback_u70_m3_ms05_s20632672_213624_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_ROS_7B.show()
fig_gantt_ROS_1kB.show()
# fig_gantt_ROS_1MB.show()
fig_gantt_FINS_7B.show()
fig_gantt_FINS_1kB.show()
fig_gantt_FINS_1MB.show()

正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
  worker 总 CPU 时间 : 19779.166 ms
    - 任务    : 19742.869 ms (99.8%)
    - Overhead: 36.297 ms (0.2%)
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
  worker 总 CPU 时间 : 20520.492 ms
    - 任务    : 20371.151 ms (99.3%)
    - Overhead: 149.342 ms (0.7%)
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
  worker 总 CPU 时间 : 18727.283 ms
    - 任务    : 18691.501 ms (99.8%)
    - Overhead: 35.782 ms (0.2%)
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
  worker 总 CPU 时间 : 18763.350 ms
    - 任务    : 18721.903 ms (99.8%)
    - Overhead: 41.447 ms (0.2%)
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
  worker 总 CPU 时间 : 18793.999 ms
    - 任务    : 18691.614 ms (99.5%)
    - Overhead: 102.384 ms (0.5%)


In [10]:
# CIE_FIFO_IPC_nuc12-noturbo_hythread
# feedback_u70_m3_ms05_s20632672_195219_preempt.csv
# feedback_u70_m3_ms05_s20632672_195219_timeline.csv
fig_util_ROS_7B = analyze_cpu_utilization(
    preempt_csv = "typical/CIE_FIFO_IPC_nuc12_7B/feedback_u70_m3_ms05_s20632672_195219_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_IPC_nuc12_7B/feedback_u70_m3_ms05_s20632672_195219_timeline.csv",
    analysis_window_ms=[50, 5000])

# CIE_FIFO_IPC_nuc12_1kB
# feedback_u70_m3_ms05_s20632672_152109_preempt.csv
# feedback_u70_m3_ms05_s20632672_152109_timeline.csv
fig_util_ROS_1kB = analyze_cpu_utilization(
    preempt_csv = "typical/CIE_FIFO_IPC_nuc12_1kB/feedback_u70_m3_ms05_s20632672_152109_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_IPC_nuc12_1kB/feedback_u70_m3_ms05_s20632672_152109_timeline.csv",
    analysis_window_ms=[50, 5000])

# CIE_FIFO_IPC_nuc12_1MB
# feedback_u70_m3_ms05_s20632672_213624_preempt.csv
# feedback_u70_m3_ms05_s20632672_213624_timeline.csv
# fig_util_ROS_1MB = analyze_cpu_utilization(
#     preempt_csv = "typical/CIE_FIFO_IPC_nuc12_1MB/feedback_u70_m3_ms05_s20632672_213624_preempt.csv",
#     timeline_csv = "typical/CIE_FIFO_IPC_nuc12_1MB/feedback_u70_m3_ms05_s20632672_213624_timeline.csv",
#     analysis_window_ms=[0, 1000])

# FINS
# feedback_u70_m3_ms05_s20632672_210928_preempt.csv
# feedback_u70_m3_ms05_s20632672_210928_timeline.csv
fig_util_FINS_7B = analyze_cpu_utilization(
    preempt_csv = "typical/FINS_7B/feedback_u70_m3_ms05_s20632672_210928_preempt.csv",
    timeline_csv = "typical/FINS_7B/feedback_u70_m3_ms05_s20632672_210928_timeline.csv",
    analysis_window_ms=[50, 5000])

# feedback_u70_m3_ms05_s20632672_211315_preempt.csv
# feedback_u70_m3_ms05_s20632672_211315_timeline.csv
fig_util_FINS_1kB = analyze_cpu_utilization(
    preempt_csv = "typical/FINS_1kB/feedback_u70_m3_ms05_s20632672_211315_preempt.csv",
    timeline_csv = "typical/FINS_1kB/feedback_u70_m3_ms05_s20632672_211315_timeline.csv",
    analysis_window_ms=[50, 5000])

# feedback_u70_m3_ms05_s20632672_213624_preempt.csv
# feedback_u70_m3_ms05_s20632672_213624_timeline.csv
fig_util_FINS_1MB = analyze_cpu_utilization(
    preempt_csv = "typical/FINS_1MB/feedback_u70_m3_ms05_s20632672_213624_preempt.csv",
    timeline_csv = "typical/FINS_1MB/feedback_u70_m3_ms05_s20632672_213624_timeline.csv",
    analysis_window_ms=[50, 5000])

fig_util_ROS_7B.show()
fig_util_ROS_1kB.show()
# fig_util_ROS_1MB.show()
fig_util_FINS_7B.show()
fig_util_FINS_1kB.show()
fig_util_FINS_1MB.show()

正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html

逐核利用率：
   CPU 1  Active  81.89%  Overhead   0.20%  Idle  17.91%   (total 4950.0 ms)
   CPU 2  Active  81.16%  Overhead   0.10%  Idle  18.73%   (total 4950.0 ms)
   CPU 3  Active  47.08%  Overhead   0.08%  Idle  52.84%   (total 4950.0 ms)

全局加权：
  Active    70.05%  (10401.9 ms / 14850.0 ms)
  Overhead   0.13%  (18.6 ms / 14850.0 ms)
  Idle      29.83%  (4429.4 ms / 14850.0 ms)
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html

逐核利用率：
   CPU 2  Active  73.44%  Overhead   0.52%  Idle  26.04%   (total 4950.0 ms)
   CPU 3  Active  69.70%  Overhead   0.58%  Idle  29.72%   (total 4950.0 ms)
   CPU 1  Active  66.95%  Overhead   0.45%  Idle  32.60%   (total 4950.0 ms)

全局加权：
  Active    70.03%  (10399.5 ms / 14850.0 ms)
  Overhead   0.51%  (76.4 ms / 14850.0 ms)
  Idle      29.45%  (4374.0 ms / 14850.0 ms)
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html

逐核利用率：
   CPU 1  Active  90.74%  Overhead   0.11%  Idle 